# 📓 Semana 21 · Dia 2 — Upload de CSV com relatório de erros

**Curso**: Especialista Databricks — Engenharia de Dados → GenAI → Agentes

| Campo | Valor |
|---|---|
| **Plano** | ✅ Free Edition (app Streamlit) |
| **Tempo estimado** | 2h |
| **Certificação alvo** | Portfólio empresarial |
| **Pré-requisitos** | SQL e Python básicos · notebooks anteriores do curso |
| **Entregável do dia** | Upload + relatório de erros |

---


## 📖 Teoria — O fluxo de upload

1. Usuário envia CSV
2. Parse e leitura (pandas)
3. Motor valida linha a linha (4 camadas)
4. Relatório: linhas OK/erro com mensagem clara
5. Salvar submissão + itens + validações
6. (Se tudo OK → fluxo de aprovação, Semana 23)


### 💻 Na prática — Upload + validação

Leia o CSV e valide por linha.


In [ ]:
# Upload e validação (pandas)
import pandas as pd
from io import StringIO
def processar_csv(conteudo, yaml_def):
    df = pd.read_csv(StringIO(conteudo))
    erros = valida_estrutura(df, yaml_def)
    if erros:
        return None, erros
    relatorio = []
    for i, row in df.iterrows():
        linha_erros = []
        for campo, cfg in {c["nome"]: c for c in yaml_def["campos"]}.items():
            if campo in df.columns:
                e = valida_tipo(row.get(campo), cfg)
                if e:
                    linha_erros.append(f"{campo}: {e}")
        if linha_erros:
            relatorio.append({"linha": i+2, "erros": linha_erros})  # +2 (header)
    return df, relatorio
print("processar_csv pronto (retorna df + relatório).")

In [ ]:
# Exemplo de relatório
relatorio_exemplo = [
    {"linha": 3, "erros": ["meta: não é número"]},
    {"linha": 5, "erros": ["mes: mês inválido (01-12)", "vendedor obrigatório"]},
]
for r in relatorio_exemplo:
    print(f"Linha {r['linha']}: " + "; ".join(r["erros"]))

### 💻 Na prática — Salvando a submissão

Persista submissão + itens + validações.


In [ ]:
# Salvar submissão
import uuid
def salvar_submissao(fluxo_id, df, relatorio, origem="csv"):
    sid = str(uuid.uuid4())
    spark.createDataFrame([(sid, fluxo_id, origem, "pendente", "now", "ana")],
        ["submissao_id", "fluxo_id", "origem", "status", "criado_em", "criado_por"])\
        .withColumn("criado_em", current_timestamp())\
        .write.mode("append").saveAsTable("workspace.app.submissoes")
    print(f"Submissão {sid} salva com {len(relatorio)} linhas com erro.")
salvar_submissao("metas", None, relatorio_exemplo)

> 🎯 **Dica de prova**: UX de validação: relatório por LINHA com mensagem clara (não 'erro genérico') — é o que faz o app ser adotado pelo negócio.


## 🎯 Exercícios de fixação

**1.** Teste o upload com um CSV de 10 linhas (2 com erro).

**2.** Mostre no app: contagem OK/erro + tabela dos erros.

**3.** O que fazer com linhas OK quando há erros? (decisão de produto)


> Tente resolver **antes** de olhar o gabarito no final do notebook.


## 🗝️ Gabarito comentado

**1.** Teste

Relatório deve apontar exatamente as linhas/campos.

**2.** App

st.file_uploader + st.dataframe do relatório + métricas OK/erro.

**3.** Decisão

Política: ou bloquear tudo, ou processar só as OK (configurável por fluxo no YAML).



## ✅ Checklist de fechamento

- [ ] Rodei todas as células do notebook do início ao fim sem erros.
- [ ] Consigo explicar os conceitos de hoje em 3 frases (sem olhar o material).
- [ ] Fiz os exercícios e conferi o gabarito.
- [ ] Anotei as dúvidas que preciso revisar.

---
*Próximo passo: siga para o notebook seguinte do plano do curso.*